### Imports

In [15]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings

import json
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

### Configs

In [16]:
# Tunable parameters
CONFIG = {
    "window_size_seconds": 120,
    "similarity_threshold_correct": 0.70,
    "similarity_threshold_weak": 0.45,
    "extended_silence_threshold": 120,
    "consecutive_incorrect_warning": 2,
    "consecutive_incorrect_severe": 3,
    "embedding_model": "all-MiniLM-L6-v2"
}

SESSION_CONTRACT = {
    "topic_name": "Basic Math",
    "scope": "beginner",
    "allowed_concepts": [
        "addition", "subtraction", "numbers", "counting", 
        "plus", "minus", "sum", "equals", "basic", "simple"
    ],
    "disallowed_concepts": [
        "stocks", "finance", "derivatives", "calculus", 
        "investment", "trading", "portfolio", "quantum", 
        "integration", "advanced", "complex"
    ]
}

print("Configuration loaded:")
print(json.dumps(CONFIG, indent=2))
print("\nSession Contract:")
print(json.dumps(SESSION_CONTRACT, indent=2))

Configuration loaded:
{
  "window_size_seconds": 120,
  "similarity_threshold_correct": 0.7,
  "similarity_threshold_weak": 0.45,
  "extended_silence_threshold": 120,
  "consecutive_incorrect_warning": 2,
  "consecutive_incorrect_severe": 3,
  "embedding_model": "all-MiniLM-L6-v2"
}

Session Contract:
{
  "topic_name": "Basic Math",
  "scope": "beginner",
  "allowed_concepts": [
    "addition",
    "subtraction",
    "numbers",
    "counting",
    "plus",
    "minus",
    "sum",
    "equals",
    "basic",
    "simple"
  ],
  "disallowed_concepts": [
    "stocks",
    "finance",
    "derivatives",
    "calculus",
    "investment",
    "trading",
    "portfolio",
    "quantum",
    "integration",
    "advanced",
    "complex"
  ]
}


### Load embedding model

In [17]:
print("Loading embedding model...")
model = SentenceTransformer(CONFIG["embedding_model"])
print("Model loaded successfully!")

Loading embedding model...
Model loaded successfully!


### Build topic reference embedding

In [18]:
def build_topic_embedding(contract):
    """Create semantic anchor for topic correctness"""
    topic_text = contract["topic_name"] + " " + " ".join(contract["allowed_concepts"])
    embedding = model.encode(topic_text)
    return embedding

topic_embedding = build_topic_embedding(SESSION_CONTRACT)
print(f"Topic embedding generated. Shape: {topic_embedding.shape}")

Topic embedding generated. Shape: (384,)


### Window building function

In [19]:
def build_windows(transcript, window_size):
    """Group transcript entries into time windows"""
    windows = []
    current_window = []
    window_start = transcript[0]["start"]

    for entry in transcript:
        if entry["end"] - window_start <= window_size:
            current_window.append(entry)
        else:
            windows.append(current_window)
            current_window = [entry]
            window_start = entry["start"]

    if current_window:
        windows.append(current_window)

    return windows

### Window analysis function

In [20]:
def analyze_window(window, topic_embedding, contract, model):
    """Analyze semantic correctness and scope of a window"""
    # Combine all text in window, filtering empty/None entries
    text_parts = []
    for entry in window:
        entry_text = entry.get("text", "").strip()
        if entry_text and entry_text not in ["[inaudible]", "[noise]", "..."]:
            text_parts.append(entry_text)
    
    text = " ".join(text_parts)
    
    # Handle empty or gibberish windows
    if len(text) < 10:  # Too short to analyze
        return {
            "text": text or "[EMPTY WINDOW]",
            "similarity": 0.0,
            "out_of_scope": False,
            "start_time": window[0]["start"],
            "end_time": window[-1]["end"],
            "error": "insufficient_content"
        }
    
    # Generate window embedding
    window_embedding = model.encode(text)
    
    # Compute similarity
    similarity = cosine_similarity(
        [window_embedding],
        [topic_embedding]
    )[0][0]
    
    # Check for disallowed concepts (scope violation)
    text_lower = text.lower()
    out_of_scope = any(
        bad_word in text_lower
        for bad_word in contract["disallowed_concepts"]
    )
    
    # Get window time range
    start_time = window[0]["start"]
    end_time = window[-1]["end"]
    
    return {
        "text": text,
        "similarity": float(similarity),
        "out_of_scope": out_of_scope,
        "start_time": start_time,
        "end_time": end_time
    }

### Classification Function

In [21]:
def classify_window(similarity, out_of_scope, config):
    """Classify window correctness status"""
    if out_of_scope:
        return "out_of_scope"
    elif similarity >= config["similarity_threshold_correct"]:
        return "correct"
    elif similarity >= config["similarity_threshold_weak"]:
        return "weakly_correct"
    else:
        return "incorrect"

### Extended Silence Detector

In [22]:
def check_extended_silence(transcript, threshold):
    """Detect gaps in conversation exceeding threshold"""
    silence_events = []
    
    for i in range(1, len(transcript)):
        gap = transcript[i]["start"] - transcript[i-1]["end"]
        if gap >= threshold:
            silence_events.append({
                "gap_seconds": gap,
                "after_timestamp": transcript[i-1]["end"],
                "before_timestamp": transcript[i]["start"]
            })
    
    return silence_events

### Warning Decision Engine

In [23]:
def generate_warning(consecutive_incorrect, window_number, reason):
    """Generate warning based on temporal patterns"""
    severity = "none"
    message = ""
    
    if consecutive_incorrect >= CONFIG["consecutive_incorrect_severe"]:
        severity = "severe"
        message = f"SEVERE WARNING: {consecutive_incorrect} consecutive incorrect windows detected. Session should be terminated."
    elif consecutive_incorrect >= CONFIG["consecutive_incorrect_warning"]:
        severity = "mild"
        message = f"WARNING: {consecutive_incorrect} consecutive incorrect windows detected. Topic mismatch."
    
    if severity != "none":
        return {
            "severity": severity,
            "message": message,
            "window_number": window_number,
            "consecutive_count": consecutive_incorrect,
            "reason": reason,
            "timestamp": datetime.now().isoformat()
        }
    
    return None

### Main Monitoring Function

In [24]:
def monitor_session(session_data, contract, topic_embedding, config, model):
    """Main session monitoring pipeline"""
    
    session_id = session_data["session_id"]
    transcript = session_data["transcript"]
    
    print(f"\n{'='*60}")
    print(f"MONITORING SESSION: {session_id}")
    print(f"Topic: {contract['topic_name']} ({contract['scope']})")
    print(f"{'='*60}\n")
    
    # Build windows
    windows = build_windows(transcript, config["window_size_seconds"])
    print(f"Total windows created: {len(windows)}\n")
    
    # Tracking variables
    consecutive_incorrect = 0
    total_incorrect = 0
    warnings = []
    window_results = []
    
    # Analyze each window
    # Analyze each window
    for i, window in enumerate(windows):
        result = analyze_window(window, topic_embedding, contract, model)
        
        # Handle insufficient content
        if "error" in result:
            print(f"Window {i+1} [{result['start_time']}s - {result['end_time']}s]")
            print(f"ERROR: {result['error'].upper()}")
            print(f"Text: {result['text']}")
            print()
            continue
        
        status = classify_window(result["similarity"], result["out_of_scope"], config)
        
        # Store result
        window_result = {
            "window_number": i + 1,
            "start_time": result["start_time"],
            "end_time": result["end_time"],
            "text_preview": result["text"][:100] + "..." if len(result["text"]) > 100 else result["text"],
            "similarity_score": round(result["similarity"], 3),
            "status": status,
            "out_of_scope": result["out_of_scope"]
        }
        window_results.append(window_result)
        
        # Console output
        print(f"Window {i+1} [{result['start_time']}s - {result['end_time']}s]")
        print(f"Similarity: {result['similarity']:.3f}")
        print(f"Status: {status.upper()}")
        if result["out_of_scope"]:
            print(f"SCOPE VIOLATION DETECTED")
        
        # Update counters
        if status in ["incorrect", "out_of_scope"]:
            consecutive_incorrect += 1
            total_incorrect += 1
        else:
            consecutive_incorrect = 0
        
        # Check for warnings
        warning = generate_warning(consecutive_incorrect, i + 1, status)
        if warning:
            warnings.append(warning)
            print(f"{warning['message']}")
        
        print()
    
    # Check for extended silence
    silence_events = check_extended_silence(transcript, config["extended_silence_threshold"])
    if silence_events:
        silence_warning = {
            "severity": "structural",
            "message": f"Extended silence detected: {len(silence_events)} gap(s) exceeding {config['extended_silence_threshold']}s",
            "events": silence_events,
            "timestamp": datetime.now().isoformat()
        }
        warnings.append(silence_warning)
        print(f"{silence_warning['message']}")
    
    # Session summary
    total_windows = len(windows)
    incorrect_percentage = (total_incorrect / total_windows * 100) if total_windows > 0 else 0
    
    summary = {
        "session_id": session_id,
        "total_windows": total_windows,
        "total_incorrect": total_incorrect,
        "incorrect_percentage": round(incorrect_percentage, 2),
        "total_warnings": len(warnings),
        "extended_silence_events": len(silence_events)
    }
    
    print(f"\n{'='*60}")
    print("SESSION SUMMARY")
    print(f"{'='*60}")
    print(f"Total windows: {total_windows}")
    print(f"Incorrect windows: {total_incorrect} ({incorrect_percentage:.1f}%)")
    print(f"Warnings issued: {len(warnings)}")
    print(f"Extended silence events: {len(silence_events)}")
    print(f"{'='*60}\n")
    
    # Post-session handoff
    handoff = {
        "session_id": session_id,
        "timestamp": datetime.now().isoformat(),
        "window_timeline": window_results,
        "warning_history": warnings,
        "structural_violations": {
            "extended_silence": silence_events,
            "late_join": session_data["session_metadata"].get("late_join", False),
            "early_leave": session_data["session_metadata"].get("early_leave", False)
        },
        "summary": summary,
        "ground_truth": session_data.get("ground_truth", {})
    }
    
    return handoff

### Save Logs Function

In [25]:
def save_session_log(handoff, output_dir="logs"):
    """Save session analysis to JSON file"""
    os.makedirs(output_dir, exist_ok=True)
    
    filename = f"{output_dir}/{handoff['session_id']}_log.json"
    
    with open(filename, 'w') as f:
        json.dump(handoff, f, indent=2)
    
    print(f"Log saved: {filename}")
    return filename

### Process all sessions

In [30]:
import glob

# Automatically find all JSON session files
session_directory = "/kaggle/input/mock-sessions-updated/"
session_files = glob.glob(f"{session_directory}*.json")
session_files.sort()  # Process in alphabetical order

print(f"Found {len(session_files)} session file(s):")
for f in session_files:
    print(f"  - {f}")
print("\n" + "="*60 + "\n")

all_handoffs = []

for session_file in session_files:
    try:
        # Load session data
        with open(session_file, 'r') as f:
            session_data = json.load(f)
        
        # Monitor session
        handoff = monitor_session(
            session_data, 
            SESSION_CONTRACT, 
            topic_embedding, 
            CONFIG, 
            model
        )
        
        # Save log
        save_session_log(handoff)
        
        all_handoffs.append(handoff)
        
        print("\n" + "="*60 + "\n")
        
    except FileNotFoundError:
        print(f"File not found: {session_file}")
    except Exception as e:
        print(f"Error processing {session_file}: {str(e)}")

print(f"\n✅ Processed {len(all_handoffs)} session(s) successfully\n")

Found 5 session file(s):
  - /kaggle/input/mock-sessions-updated/session_001_good.json
  - /kaggle/input/mock-sessions-updated/session_002_drift.json
  - /kaggle/input/mock-sessions-updated/session_003_silence.json
  - /kaggle/input/mock-sessions-updated/session_004_mixed.json
  - /kaggle/input/mock-sessions-updated/session_005_gibberish.json



MONITORING SESSION: S001
Topic: Basic Math (beginner)

Total windows created: 4

Window 1 [0s - 120s]
Similarity: 0.715
Status: CORRECT

Window 2 [120s - 210s]
Similarity: 0.454
Status: WEAKLY_CORRECT

Window 3 [210s - 300s]
Similarity: 0.628
Status: WEAKLY_CORRECT

Window 4 [300s - 360s]
Similarity: 0.484
Status: WEAKLY_CORRECT


SESSION SUMMARY
Total windows: 4
Incorrect windows: 0 (0.0%)
Warnings issued: 0
Extended silence events: 0

Log saved: logs/S001_log.json



MONITORING SESSION: S002
Topic: Basic Math (beginner)

Total windows created: 3

Window 1 [0s - 120s]
Similarity: 0.721
Status: CORRECT

Window 2 [120s - 240s]
Similarity: 0.206


### Aggregate Analysis

In [27]:
print("AGGREGATE ANALYSIS ACROSS ALL SESSIONS")
print("="*60)

for handoff in all_handoffs:
    summary = handoff["summary"]
    ground_truth = handoff.get("ground_truth", {})
    warnings = handoff["warning_history"]
    
    print(f"\n{handoff['session_id']}:")
    print(f"  Incorrect: {summary['incorrect_percentage']:.1f}%")
    print(f"  Warnings: {summary['total_warnings']}")
    print(f"  Ground Truth: {ground_truth.get('final_outcome', 'N/A')}")
    
    # Enhanced Model Verdict Logic
    model_verdict = "acceptable"
    
    # Check for severe warnings
    has_severe_warning = any(w.get("severity") == "severe" for w in warnings)
    
    # Decision rules
    if has_severe_warning or summary['incorrect_percentage'] >= 60:
        model_verdict = "severe_violation"
    elif summary['total_warnings'] >= 1 or summary['incorrect_percentage'] >= 30:
        model_verdict = "warning_worthy"
    
    print(f"  Model Verdict: {model_verdict}")
    match = "OK" if model_verdict == ground_truth.get('final_outcome', '') else "NO"
    print(f"  Match: {match}")

print("\n" + "="*60)

AGGREGATE ANALYSIS ACROSS ALL SESSIONS

S001:
  Incorrect: 0.0%
  Warnings: 0
  Ground Truth: acceptable
  Model Verdict: acceptable
  Match: OK

S002:
  Incorrect: 66.7%
  Warnings: 1
  Ground Truth: severe_violation
  Model Verdict: severe_violation
  Match: OK

S003:
  Incorrect: 100.0%
  Warnings: 2
  Ground Truth: severe_violation
  Model Verdict: severe_violation
  Match: OK

S004:
  Incorrect: 66.7%
  Warnings: 1
  Ground Truth: warning_worthy
  Model Verdict: severe_violation
  Match: NO

S005:
  Incorrect: 33.3%
  Warnings: 0
  Ground Truth: warning_worthy
  Model Verdict: warning_worthy
  Match: OK



### Export Summary Report

In [28]:
# Create final summary report
summary_report = {
    "analysis_timestamp": datetime.now().isoformat(),
    "configuration": CONFIG,
    "session_contract": SESSION_CONTRACT,
    "total_sessions_analyzed": len(all_handoffs),
    "sessions": [
        {
            "session_id": h["session_id"],
            "summary": h["summary"],
            "ground_truth": h.get("ground_truth", {}),
            "warnings_count": len(h["warning_history"])
        }
        for h in all_handoffs
    ]
}

with open("logs/summary_report.json", 'w') as f:
    json.dump(summary_report, f, indent=2)

print("Summary report saved: logs/summary_report.json")

Summary report saved: logs/summary_report.json
